# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install matplotlib scipy --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import matplotlib.pyplot as plt
import numpy as np
import math
from scipy.stats import binom

### 1. Access Protocols

There are three basic access protocols: (Pure) ALOHA, Slotted ALOHA, and Carrier Sense Multiple Access. 

#### 1.1 ALOHA

ALOHA was first introduced by the University of Hawaii; the abbreviation has no meaning in the context of access protocols. The first ALOHA network, developed in the early 1970s, connected computers across the Hawaiian islands Oahu, Maui, Hawaii, and Kauai.  The ALOHA system used radio communication to enable these islands to share data, making it one of the earliest implementations of a wireless network.

ALOHA is a simple channel access protocol designed for wireless communication systems, particularly in shared or broadcast channels. It allows devices to transmit data whenever they have data to send, without any coordination between devices.
 - A device sends data as soon as it is ready, without checking if the channel is busy.
 - After transmitting, the device waits for an acknowledgment (ACK) from the receiver.
 - If an ACK is received, the transmission was successful. If not (due to a collision with another device's transmission), the device waits a random period of time before retransmitting.
   
The main advantage is the implementation simplicity. Disadvantages are the high probability of collision and low efficiency. Since devices transmit whenever they want, multiple devices may send data at the same time, resulting in collisions. The maximum throughput (successful transmission rate) is limited to about 18.4%, meaning a lot of transmission time is wasted on collisions.

In [ ]:
# --- Pure ALOHA Protocol ---
def pure_aloha(inter_arrival_time: np.ndarray):
    success = 0
    for j in range(len(inter_arrival_time)):
        if j - 1 >= 0 and inter_arrival_time[j - 1] >= 1 and j + 1 < N and inter_arrival_time[j + 1] >= 1:
            success += 1
    return success

#### 1.2 Slotted ALOHA
Slotted ALOHA improves upon the ALOHA protocol by dividing time into discrete slots. Devices can only transmit at the beginning of a time slot, reducing the chance of collisions.
 - Time is divided into uniform slots equal to the time it takes to send one data packet.
 - Devices can only transmit data at the start of a time slot.
 - If two or more devices attempt to transmit in the same slot, a collision occurs, and the data is lost.
 - After a collision, each device waits for a random number of slots before retransmitting.
   
Advantages of Slotted ALOHA over Pure ALOHA are the lower collision probability and the higher efficiency. Since transmissions are restricted to time slots, the likelihood of two devices transmitting at exactly the same time is reduced compared to pure ALOHA. Slotted ALOHA can achieve a maximum throughput of around 37%, which is twice that of pure ALOHA. The main disadvantage is the need for time synchronization among the devices. Devices must synchronize to a common time slot structure, which adds complexity.


In [ ]:
# --- Slotted ALOHA Protocol ---
def slotted_aloha(arrival_times: np.ndarray):
    count_arrivals = np.zeros(math.ceil(arrival_times[-1]) + 1)
    for val in arrival_times:
        count_arrivals[math.ceil(val)] += 1
    return np.sum(1 * (count_arrivals == 1))

#### 1.3 CSMA (Carrier Sense Multiple Access)
CSMA (Carrier Sense Multiple Access) is a more sophisticated protocol that reduces the likelihood of collisions by requiring devices to sense or check the status of the channel before transmitting. It operates under the principle of "listen before talk."
 - Before a device transmits, it checks if the channel is busy.
 - If the channel is idle (no one else is transmitting), the device begins transmission.
 - If the channel is busy, the device waits until it becomes idle.
 - If two or more devices sense that the channel is idle at the same time and transmit simultaneously, a collision occurs, which will require a retransmission.

CSMA can be implemented in several variations. CSMA with Collision Detection (CSMA/CD) is used in Ethernet. In CSMA/CD, devices monitor the channel while transmitting to detect if a collision has occurred. If a collision is detected, the transmission stops immediately, and the device waits a random amount of time before attempting to retransmit. The other variation is CSMA with Collision Avoidance (CSMA/CA), which is predominantly used in Wi-Fi (IEEE 802.11). Instead of detecting collisions (which is hard in wireless systems), devices take steps to avoid collisions by using random backoff. After sensing the channel is idle, the device waits a random period before transmitting to avoid simultaneous transmissions. Receivers also send Acknowledgements (ACKs) for successful transmissions. If an ACK is not received, the transmitter assumes a collision occurred and retransmits.


The advantage of CSMA is that it is more efficient than ALOHA. By sensing the channel and waiting for it to be idle, CSMA reduces the likelihood of collisions and wasted transmission attempts. Collision detection/avoidance further reduces inefficiencies.

Disadvantages are the delay and complexity. There is still a possibility of collisions if two or more devices sense the channel as idle at the same time due to propagation delay (the time it takes for a signal to travel through the network). CSMA is more complex than ALOHA because of backoff or retransmission strategies. CSMA requires devices to monitor the channel.


In [ ]:
# --- CSMA Protocol ---
def csma_generalized(arrival_times: np.ndarray, p, cd: bool = False):
    beta = .1
    success = 0
    transmission_ends = 0
    collision_detect_possible = 0
    successfully_sending = False
    pending_packets = 0
    for j, time in enumerate(arrival_times):
        if time < collision_detect_possible:
            # channel is busy, but detected as free due to beta delay in detection of ongoing transmission
            if cd:
                transmission_ends = min(transmission_ends, time + beta)
            else:
                transmission_ends = time + 1
            if successfully_sending:
                success -= 1
                successfully_sending = False
        elif time < transmission_ends:
            # channel is busy, do nothing
            pending_packets += 1
            if (j + 1 < len(arrival_times) and arrival_times[j + 1] >= transmission_ends) \
                    or j + 1 == len(arrival_times):
                if p == 1:
                    x = pending_packets
                elif p == 0:
                    x = 0
                else:
                    x = binom(pending_packets, p).rvs()
                if x == 1:
                    success += 1
                    successfully_sending = True
                    collision_detect_possible = transmission_ends + beta
                    transmission_ends = transmission_ends + 1
                elif x > 1:
                    collision_detect_possible = transmission_ends + beta
                    transmission_ends = transmission_ends + 1
                pending_packets = 0
        else:
            # channel is free
            successfully_sending = True
            success += 1
            collision_detect_possible = time + beta
            transmission_ends = time + 1
    return success

### 2. Simulation

In [ ]:
# --- Simulation ---

# Parameters
N = int(10000)
numG = 30
rng = np.random.default_rng()

# Initialize throughput arrays
throughput_pure = np.zeros(numG)
throughput_slotted = np.zeros(numG)
throughput_csma_non_persistent = np.zeros(numG)
throughput_1persistent = np.zeros(numG)
throughput_pt1persistent = np.zeros(numG)
throughput_pt01persistent = np.zeros(numG)
throughput_pt5persistent = np.zeros(numG)
throughput_csma_cd = np.zeros(numG)
G = np.logspace(-1.5, .9, numG)

# Run simulation for different offered loads G
for i, lam in enumerate(G):
    inter_arrival_time = rng.exponential(1 / lam, N)
    arrival_times = np.array(inter_arrival_time)

    # Compute cumulative arrival times
    for j in range(1, len(arrival_times)):
        arrival_times[j] += arrival_times[j - 1]

    # Calculate throughput for each protocol
    throughput_pure[i] = pure_aloha(inter_arrival_time) / math.ceil(arrival_times[-1])
    throughput_slotted[i] = slotted_aloha(arrival_times) / math.ceil(arrival_times[-1])
    throughput_csma_non_persistent[i] = csma_generalized(arrival_times, 0) / math.ceil(arrival_times[-1])
    throughput_1persistent[i] = csma_generalized(arrival_times, 1) / math.ceil(arrival_times[-1])
    throughput_pt1persistent[i] = csma_generalized(arrival_times, .1) / math.ceil(arrival_times[-1])
    throughput_pt01persistent[i] = csma_generalized(arrival_times, .01) / math.ceil(arrival_times[-1])
    throughput_pt5persistent[i] = csma_generalized(arrival_times, .5) / math.ceil(arrival_times[-1])
    throughput_csma_cd[i] = csma_generalized(arrival_times, 1, True) / math.ceil(arrival_times[-1])

In [ ]:
# --- Plotting Results of all Protocols ---
fig, ax = plt.subplots(figsize=(8, 5))
theLineWidth = 2

ax.cla()
ax.axis([None, None, None, None])

# Plot results
ax.plot(G, throughput_pure, linewidth=theLineWidth+2, label='Pure Aloha')
ax.plot(G, throughput_slotted, linewidth=theLineWidth+2, label='Slotted Aloha')
ax.plot(G, throughput_csma_non_persistent, linewidth=theLineWidth, label="CSMA non-persistent")
ax.plot(G, throughput_1persistent, linewidth=theLineWidth, label="CSMA 1-persistent")
ax.plot(G, throughput_pt1persistent, linewidth=theLineWidth, label="CSMA .1-persistent")
ax.plot(G, throughput_pt5persistent, linewidth=theLineWidth, label="CSMA .5-persistent")
ax.plot(G, throughput_pt01persistent, linewidth=theLineWidth, label="CSMA .01-persistent")
ax.plot(G, throughput_csma_cd, linewidth=theLineWidth, label="CSMA Collision Detection")

# Customize plot
ax.set_xlabel('G (attempts rate)',color="white")
ax.set_ylabel('S (throughput per frame time)',color="white")

ax.tick_params(axis='x', colors='white')  # Change x-axis tick label color
ax.tick_params(axis='y', colors='white')  # Change y-axis tick label color
fig.patch.set_facecolor('#3e4f58')  
ax.grid(True)

# Customize spines
ax.spines['top'].set_color('white')
ax.spines['bottom'].set_color('white')
ax.spines['left'].set_color('white')
ax.spines['right'].set_color('white')

ax.spines['top'].set_linewidth(1)
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)
ax.spines['right'].set_linewidth(1)

ax.set_facecolor("#8998a1")
ax.legend(fancybox=True, framealpha=1, fontsize=8,
          loc="center right", facecolor=(.6, 0.61, 0.8))
plt.tight_layout()

#### 1.4 Comparison of ALOHA, Slotted ALOHA, and CSMA:

| **Protocol**    | **Channel Access Method**                     | **Collision Handling**      | **Efficiency**  | **Applications**                 |
|-----------------|-----------------------------------------------|-----------------------------|-----------------|-----------------------------------|
| **ALOHA**       | Transmit whenever data is ready               | Retransmit after collisions  | Low (~18.4%)    | Early wireless systems, satellites|
| **Slotted ALOHA** | Transmit at the beginning of time slots       | Retransmit after collisions  | Moderate (~37%) | TDMA systems, RFID               |
| **CSMA**        | Sense channel before transmitting             | Collision detection or avoidance (CD/CA) | High (better than ALOHA) | Ethernet (CSMA/CD), Wi-Fi (CSMA/CA)|